# 5. An SN2 transition-state search
**Kernel:** AIMNet2, with Sella installed. Complete the README preflight and use a fresh kernel.

**Learning goals:** construct a transition-state guess from a constrained scan; refine a saddle;
check the unstable vibrational mode; optionally follow both reaction-path directions.

![SN2 reaction](SN2.png)

We model **Br- + CH3Cl → CH3Br + Cl-**, with total charge **-1**. This is a gas-phase model
calculation. Solvent effects and free-energy corrections are outside this exercise.
The supplied older trajectories used different settings and are not validation of this revised calculation.

**Predict:** which bond forms and which bond breaks? What motion should the unstable mode show?


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository, workshop folder, or exercise folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
WORKSHOP = next((p for base in _candidates for p in (base, base / 'workshop_demo')
                 if (p / 'workshop_utils.py').is_file()), None)
if WORKSHOP is None:
    raise RuntimeError('Launch Jupyter from the repository or workshop_demo folder.')
if str(WORKSHOP) not in sys.path:
    sys.path.insert(0, str(WORKSHOP))
from workshop_utils import start_exercise, mace_model, relax, smoke_check, signed_angle
DATA, OUTPUT = start_exercise('TransitionStates')
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view


## 1. Check the input and electronic state
The six-atom XYZ uses C=0, Cl=1, Br=5. The neutral setting from the original demonstration has been
changed to -1 for the stated anionic reaction. This changes the potential-energy surface and requires
fresh calculations; never reuse the original vibration cache.


In [ ]:
from aimnet.calculators import AIMNet2ASE
from ase.optimize import BFGS
from ase.constraints import FixAtoms
calculator = AIMNet2ASE('aimnet2-2025', charge=-1)
molecule = read(DATA / 'sn2_initial_structure.xyz')
molecule.calc = calculator
print(list(enumerate(molecule.get_chemical_symbols())))
smoke_check(molecule)
view(molecule, viewer='x3d')


## 2. Scan the approaching C-Br distance
Fix C and Br at each separation and relax the other atoms. This is a constrained path used to find
a guess, not proof of a minimum-energy reaction path. A scan may fail to show an interior maximum;
if so, inspect the geometries and adjust the scan instead of selecting an endpoint automatically.


In [ ]:
res = []
for distance in np.linspace(3.0, 1.7, 14):
    molecule.set_constraint()
    molecule.set_distance(0, 5, float(distance), fix=0)
    molecule.set_constraint(FixAtoms(indices=[0, 5]))
    ok = relax(molecule, BFGS, fmax=0.03, steps=150,
               logfile=str(OUTPUT / f'scan_{distance:.2f}.log'))
    if not ok:
        raise RuntimeError(f'Scan did not converge at {distance:.2f} A; inspect before proceeding.')
    res.append((distance, molecule.get_potential_energy(), molecule.copy()))
distances = np.array([r[0] for r in res])
energies = np.array([r[1] for r in res])
write(OUTPUT / 'scan.extxyz', [r[2] for r in res])
plt.plot(distances, energies-energies[0], 'o-')
plt.xlabel('C-Br distance (angstrom)'); plt.ylabel('Energy relative to first scan point (eV)')
plt.show()


## 3. Refine an interior maximum with Sella
The highest interior local maximum is used as a starting guess. Remove the scan constraints before
saddle optimization. Convergence of this optimizer is necessary, but does not establish that the
saddle connects the intended reactant and product.


In [ ]:
from sella import Sella
maxima = [i for i in range(1, len(energies)-1)
          if energies[i] > energies[i-1] and energies[i] > energies[i+1]]
if not maxima:
    raise RuntimeError('No interior maximum found. Inspect the scan and revise its range or guess.')
guess_index = max(maxima, key=lambda i: energies[i])
ts = res[guess_index][2].copy()
ts.set_constraint()
ts.calc = calculator
opt = Sella(ts, internal=True, logfile=str(OUTPUT / 'sella.log'),
            trajectory=str(OUTPUT / 'sella.traj'))
ts_ok = bool(opt.run(fmax=0.01, steps=200))
print('Sella converged:', ts_ok)
write(OUTPUT / 'ts_candidate.extxyz', ts)
if not ts_ok:
    raise RuntimeError('TS step limit reached. Inspect the trajectory before running vibrations.')
view(ts, viewer='x3d')


## 4. Validate the saddle using vibrations
For an unconstrained nonlinear molecule, expect one chemically meaningful imaginary frequency at
a first-order saddle, plus near-zero translation/rotation modes. Small imaginary frequencies can be
numerical artifacts: the threshold below is a screening aid, not a physical definition.
Inspect the displacement: does it form C-Br while breaking C-Cl?

This finite-difference calculation makes multiple force evaluations. Its cache lives in this run's
fresh output directory. If you change the geometry, rerun setup and the calculation in a new directory.


In [ ]:
from ase.vibrations import Vibrations
vib = Vibrations(ts, name=str(OUTPUT / 'vib'))
vib.run()
vib.summary(log=str(OUTPUT / 'frequencies.txt'))
frequencies = vib.get_frequencies()
print('Frequencies (cm^-1):', frequencies)
IMAGINARY_THRESHOLD = 30.0
imaginary = np.flatnonzero(np.abs(frequencies.imag) > IMAGINARY_THRESHOLD)
print('Imaginary modes above screening threshold:', imaginary.tolist())
if len(imaginary) != 1:
    print('Do not accept the TS yet: inspect modes and numerical convergence.')
else:
    mode_index = int(imaginary[0])
    displacement = vib.get_mode(mode_index)
    for i in [0, 1, 5]:
        print(i, ts[i].symbol, 'mode displacement:', displacement[i])
    vib.write_mode(mode_index)
    mode_frames = read(OUTPUT / f'vib.{mode_index}.traj', index=':')
    FRAME = 5  # Change to inspect a different phase of the vibration.
    display(view(mode_frames[FRAME], viewer='x3d'))


## 5. Optional: follow both IRC branches
Enable only after inspecting the saddle and unstable mode. Each direction starts from a fresh copy
of the saddle and writes to its own trajectory; we do not infer a branch boundary from an energy jump.
Forward and reverse labels do not inherently identify reactants and products. Inspect both endpoints.
This extension can take substantially longer than the scan.


In [ ]:
RUN_IRC = False
if RUN_IRC:
    if len(imaginary) != 1:
        raise RuntimeError('Resolve the vibration check before the IRC extension.')
    from sella import IRC
    for direction in ['forward', 'reverse']:
        branch = ts.copy()
        branch.calc = calculator
        irc = IRC(branch, trajectory=str(OUTPUT / f'irc_{direction}.traj'),
                  logfile=str(OUTPUT / f'irc_{direction}.log'), dx=0.1, eta=1e-4, gamma=0.4)
        ok = irc.run(fmax=0.1, steps=200, direction=direction)
        print(direction, 'converged:', ok)
        print('Endpoint C-Br / C-Cl:', branch.get_distance(0, 5), branch.get_distance(0, 1))
        display(view(branch, viewer='x3d'))
else:
    print('IRC skipped. Set RUN_IRC=True for the optional extension.')


## Try, explain, and report
1. Does the unstable mode describe the expected bond exchange, rather than a global translation?
2. If you ran IRC, do the endpoints correspond to the intended reactant and product sides?
3. Why is the scan maximum minus the first scan point not automatically an activation free energy?
4. What additional evidence would you want before trusting the model for this reaction?

**Checkpoint:** record the charge, model, convergence flag, imaginary frequencies, and endpoint evidence.
A converged saddle on an MLIP surface is not by itself validation against electronic-structure reference data.
See [Sella](https://github.com/zadorlab/sella) and [ASE vibrations](https://docs.ase-lib.org/ase/vibrations/modes.html).
